In [1]:
#Bibliotecas a utilizar en este código
import pandas as pd # para la manipulación de tablas 
import geopandas as gpd #para la manipulación de objetos vectoriales
import matplotlib.pyplot as plt #para la creación de gráfica
#import gfwapiclient as gfw se explica más adelante porque es el objetivo del script


In [2]:
%pip install gfw-api-python-client

Note: you may need to restart the kernel to use updated packages.


In [3]:
#Biblioteca de GFW
import gfwapiclient as gfw

print("Cliente oficial de GFW instalado correctamente")

Cliente oficial de GFW instalado correctamente


In [4]:
#Importa la función "getpass" para la incorporación de información confidencial y que no se muestre al escribir
from getpass import getpass

In [5]:
GFW_TOKEN = getpass("Introduce tu API token de GFW: ")

Introduce tu API token de GFW:  ········


In [6]:
#Con esta función se crea un objeto que permite la comunicación con el servidor de Global Fishing Watch
gfw_client = gfw.Client(
    access_token=GFW_TOKEN
)


print("Conexión inicializada")

Conexión inicializada


In [8]:
#Con esta función se crea un objeto que permite la comunicación con el servidor de Global Fishing Watch
gfw_client = gfw.Client(
    access_token=GFW_TOKEN
)


print("Conexión inicializada")

Conexión inicializada


In [9]:
import os

# Identificar el directorio de trabajo
print("Directorio actual:")
print(os.getcwd())

# Mostrar los archivos disponibles
print("\nArchivos disponibles:")
print(os.listdir())

Directorio actual:
/home/jovyan/Tareas

Archivos disponibles:
['.ipynb_checkpoints', 'Tarea-1.ipynb', 'SATsum_diaria_2dr_2025_zee_caribbean_2026-11-09_1332.csv', 'Tarea-2.ipynb', 'Proyecto-final.ipynb', 'Prueba-3-GFW.ipynb', 'Shiretoko_EBSA_10401_10402.qmd', 'Shiretoko_EBSA_10401_10402.geojson']


In [10]:
#Cambiar el directorio de trabajo
import os

os.chdir("/home/jovyan/Tareas")

print(os.getcwd())

/home/jovyan/Tareas


In [11]:
import json

# Cargar el archivo GeoJSON
with open("Shiretoko_EBSA_10401_10402.geojson", "r", encoding="utf-8") as archivo:
    shiretoko = json.load(archivo)

In [12]:
import os

print([f for f in os.listdir() if f.endswith(".geojson")])

['Shiretoko_EBSA_10401_10402.geojson']


In [13]:
# Cargar el archivo GeoJSON
with open("Shiretoko_EBSA_10401_10402.geojson", "r", encoding="utf-8") as archivo:
    shiretoko = json.load(archivo)

# Revisar el contenido
print("Tipo de archivo:", shiretoko["type"])
print("Número de elementos:", len(shiretoko["features"]))

# Mostrar información de cada polígono
for feature in shiretoko["features"]:
    print("Área:", feature["properties"]["AreaNumber"])
    print("Nombre:", feature["properties"]["AreaName"])
    print("Geometría:", feature["geometry"]["type"])
    print("-------------------------")

Tipo de archivo: FeatureCollection
Número de elementos: 2
Área: 10401.0
Nombre: Shiretoko (Shari, Utoro) side coast
Geometría: MultiPolygon
-------------------------
Área: 10402.0
Nombre: Shiretoko (Rausu) side coast
Geometría: MultiPolygon
-------------------------


In [14]:
# Crear una lista vacía para almacenar las coordenadas
coordenadas = []

# Recorrer los dos elementos del GeoJSON
for feature in shiretoko["features"]:

    geometria = feature["geometry"]

    if geometria["type"] == "MultiPolygon":
        coordenadas.extend(geometria["coordinates"])

    elif geometria["type"] == "Polygon":
        coordenadas.append(geometria["coordinates"])

    else:
        raise ValueError(
            f"Geometría no compatible: {geometria['type']}"
        )

# Construir una geometría MultiPolygon
shiretoko_geojson = {
    "type": "MultiPolygon",
    "coordinates": coordenadas
}

# Comprobar el resultado
print("Tipo de geometría:", shiretoko_geojson["type"])
print("Número de polígonos:", len(shiretoko_geojson["coordinates"]))

Tipo de geometría: MultiPolygon
Número de polígonos: 2


In [15]:
# Consulta de esfuerzo pesquero en las áreas EBSA de Shiretoko

GFW_SHIRETOKO = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    temporal_resolution="MONTHLY",
    group_by="FLAG",
    start_date="2022-01-01",
    end_date="2023-01-01",
    geojson=shiretoko_geojson
)

# Convertir los resultados a un DataFrame
GFW_SHIRETOKO_DF = GFW_SHIRETOKO.df()

# Mostrar los primeros registros
GFW_SHIRETOKO_DF.head()

,date,detections,flag,gear_type,hours,vessel_ids,vessel_id,vessel_type,entry_timestamp,exit_timestamp,first_transmission_date,last_transmission_date,imo,mmsi,call_sign,dataset,report_dataset,ship_name,lat,lon
0,2022-07,None,JPN,None,1.816111,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v5.0,None,44.4,145.4
1,2022-12,None,JPN,None,4.265556,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v5.0,None,44.1,145.1
2,2022-11,None,JPN,None,0.749722,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v5.0,None,44.1,145.0
3,2022-10,None,JPN,None,4.592778,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v5.0,None,44.1,145.0
4,2022-12,None,JPN,None,5.810833,1,None,None,None,None,None,None,None,None,None,None,public-global-fishing-effort:v5.0,None,44.2,145.1


In [16]:
# Revisar las dimensiones de la tabla
print("Dimensiones:", GFW_SHIRETOKO_DF.shape)

Dimensiones: (11, 20)


In [17]:
# Mostrar todas las columnas
print("\nColumnas disponibles:")
print(GFW_SHIRETOKO_DF.columns.tolist())


Columnas disponibles:
['date', 'detections', 'flag', 'gear_type', 'hours', 'vessel_ids', 'vessel_id', 'vessel_type', 'entry_timestamp', 'exit_timestamp', 'first_transmission_date', 'last_transmission_date', 'imo', 'mmsi', 'call_sign', 'dataset', 'report_dataset', 'ship_name', 'lat', 'lon']


In [18]:
# Revisar tipos de datos y valores faltantes
print("\nInformación general:")
GFW_SHIRETOKO_DF.info()


Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     11 non-null     object 
 1   detections               0 non-null      object 
 2   flag                     11 non-null     object 
 3   gear_type                0 non-null      object 
 4   hours                    11 non-null     float64
 5   vessel_ids               11 non-null     int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_s